# 🔑 Solutions: Lesson 03 · Forward Propagation & Loss Functions

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 3.1: MSE, MAE & Huber  &nbsp; `⭐`

Implement `mse(y_hat, y)`, `mae(y_hat, y)` and `huber(y_hat, y, delta=1.0)` (all return the **mean** over the samples).


In [2]:
def mse(y_hat, y):
    return np.mean((y_hat - y) ** 2)
def mae(y_hat, y):
    return np.mean(np.abs(y_hat - y))
def huber(y_hat, y, delta=1.0):
    e = np.abs(y_hat - y)
    return np.mean(np.where(e <= delta, 0.5 * e ** 2, delta * (e - 0.5 * delta)))

In [3]:
# ▶ self-check
yh, y = np.array([1., 2., 3., 10.]), np.array([1., 2., 4., 4.])
try:
    dl.check("MSE = 9.25", np.isclose(mse(yh, y), 9.25))
    dl.check("MAE = 1.75", np.isclose(mae(yh, y), 1.75))
    dl.check("Huber(δ=1) = 1.5", np.isclose(huber(yh, y), (0 + 0 + .5 + 5.5) / 4))
    dl.check("Huber is less outlier-sensitive than MSE", huber(yh, y) < mse(yh, y))
except Exception:
    dl.check("implement all three losses", False)

✅ MSE = 9.25
✅ MAE = 1.75
✅ Huber(δ=1) = 1.5
✅ Huber is less outlier-sensitive than MSE


**Why this works.** The outlier (10 vs 4) contributes 36 to MSE's sum but only 5.5 to Huber's; Huber caps the influence of outliers.

---
### 🧩 Problem 3.2: Binary cross-entropy  &nbsp; `⭐⭐`

Implement `bce(p, y, eps=1e-12)`. Clip probabilities into `[eps, 1-eps]` first so `log(0)` can never happen. Return the mean loss.


In [4]:
def bce(p, y, eps=1e-12):
    p = np.clip(p, eps, 1 - eps)
    return np.mean(-(y * np.log(p) + (1 - y) * np.log(1 - p)))

In [5]:
# ▶ self-check
try:
    dl.check("perfect predictions → loss ≈ 0", bce(np.array([1., 0.]), np.array([1, 0])) < 1e-9)
    dl.check("p=0.5 everywhere → ln 2", np.isclose(bce(np.full(4, .5), np.array([1, 0, 1, 0])), np.log(2)))
    dl.check("worked example 0.572 → 0.559", np.isclose(bce(np.array([0.572]), np.array([1])), 0.5586, atol=1e-3))
    dl.check("confidently wrong is finite but huge", 20 < bce(np.array([0.0]), np.array([1])) < 40)
except Exception:
    dl.check("implement bce", False)

✅ perfect predictions → loss ≈ 0
✅ p=0.5 everywhere → ln 2
✅ worked example 0.572 → 0.559
✅ confidently wrong is finite but huge


---
### 🧩 Problem 3.3: Cross-entropy from logits  &nbsp; `⭐⭐`

Implement `ce_loss(logits, y)` for a batch: `logits` is `(m, k)`, `y` holds integer labels. Use the **log-sum-exp trick** so it works for logits like 1000.
Return the mean loss.


In [6]:
def ce_loss(logits, y):
    m = logits.max(axis=1, keepdims=True)
    lse = m[:, 0] + np.log(np.exp(logits - m).sum(axis=1))
    return np.mean(lse - logits[np.arange(len(y)), y])

In [7]:
# ▶ self-check
logits = np.array([[2.0, 1.0, 0.1], [0.1, 0.2, 3.0]]); y = np.array([0, 2])
try:
    dl.check("matches textbook value", np.isclose(ce_loss(logits, y), 0.2633, atol=1e-3))
    dl.check("stable for huge logits", np.isfinite(ce_loss(np.array([[1000., 0, -1000]]), np.array([2]))))
    dl.check("uniform logits give ln k", np.isclose(ce_loss(np.zeros((5, 10)), np.zeros(5, int)), np.log(10)))
except Exception:
    dl.check("implement ce_loss", False)

✅ matches textbook value
✅ stable for huge logits
✅ uniform logits give ln k


---
### 🧩 Problem 3.4: Forward pass + loss of the irrigation network  &nbsp; `⭐⭐`

Using the network from Section 1 (weights are provided in the starter), write `predict_and_loss(x, y)` that returns a tuple `(y_hat, loss)`:
the sigmoid output and the BCE loss: for an input `x` of shape `(2,)` and a label `y`.
Then compute the result for `x = (3, 0.5)`, `y = 0`.


In [8]:
W1 = np.array([[0.5, -0.4], [0.3, 0.8]]); b1 = np.array([0.1, -0.2])
W2 = np.array([0.7, -0.6]);               b2 = 0.05

def predict_and_loss(x, y):
    h = np.maximum(0, x @ W1 + b1)
    y_hat = 1 / (1 + np.exp(-(h @ W2 + b2)))
    loss = -(y * np.log(y_hat) + (1 - y) * np.log(1 - y_hat))
    return y_hat, loss

In [9]:
# ▶ self-check
res = predict_and_loss(np.array([1., 2.]), 1)
dl.check("worked example: ŷ≈0.572", res is not None and np.isclose(res[0], 0.572, atol=1e-3))
dl.check("worked example: loss≈0.559", res is not None and np.isclose(res[1], 0.5587, atol=1e-3))
res2 = predict_and_loss(np.array([3., 0.5]), 0)
dl.check("second example runs and loss > 0", res2 is not None and res2[1] > 0)

✅ worked example: ŷ≈0.572
✅ worked example: loss≈0.559
✅ second example runs and loss > 0


**Why this works.** A single forward pass is just a few matrix operations. The *loss* then reduces a whole prediction to the one number training will try to shrink.

---
### 🧩 Problem 3.5: Initial-loss sanity check on real data  &nbsp; `⭐⭐`

Create a randomly initialised 784 → 64 → 10 network with **small** weights (`np.random.default_rng(0).normal(0, 0.01, ...)`, zero biases),
run MNIST images (first 1,000 training images, flattened) through it with ReLU, and compute the mean cross-entropy loss using the stable formula.
Store it in `initial_loss`. It should be close to ln 10 ≈ 2.30. *(This is the first thing real practitioners check!)*


In [10]:
x_tr, y_tr, *_ = dl.load_mnist(n_train=1000, flatten=True)
rng = np.random.default_rng(0)
W1, b1 = rng.normal(0, 0.01, (784, 64)), np.zeros(64)
W2, b2 = rng.normal(0, 0.01, (64, 10)), np.zeros(10)
logits = np.maximum(0, x_tr @ W1 + b1) @ W2 + b2
m = logits.max(1, keepdims=True)
lse = m[:, 0] + np.log(np.exp(logits - m).sum(1))
initial_loss = np.mean(lse - logits[np.arange(len(y_tr)), y_tr])

In [11]:
# ▶ self-check
dl.check("initial loss is close to ln(10)=2.303", initial_loss is not None and abs(initial_loss - np.log(10)) < 0.1,
         "if you got ~50 or ~0.01 something is off: check your normalisation / formula")

✅ initial loss is close to ln(10)=2.303


**Why this works.** Small weights → near-equal logits → p ≈ 0.1 for every digit → loss ≈ −ln(0.1) = ln 10. A very different value signals a bug before you waste an hour training.